In [1]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
from google.colab import files

In [3]:
np.random.seed(42)
tf.random.set_seed(42)

In [4]:
class_names=['still','left_tilt','right_tilt',"shake"]
label_map={str(i): class_name for i, class_name in enumerate(class_names)}

In [5]:
print(label_map)

{'0': 'still', '1': 'left_tilt', '2': 'right_tilt', '3': 'shake'}


In [6]:
window_size=20
samples_per_class=1000

In [7]:
features=[
    'ax_mean', 'ax_std', 'ax_min','axmax',
    'ay_mean', 'ay_std', 'ay_min','aymax',
    'az_mean', 'az_std', 'az_min','azmax',
]

In [8]:
print("Classes:", class_names)
print("Window size:", window_size)
print("Samples per class:", samples_per_class)
print("Features:", features)

Classes: ['still', 'left_tilt', 'right_tilt', 'shake']
Window size: 20
Samples per class: 1000
Features: ['ax_mean', 'ax_std', 'ax_min', 'axmax', 'ay_mean', 'ay_std', 'ay_min', 'aymax', 'az_mean', 'az_std', 'az_min', 'azmax']


In [9]:
def generate_accel_window(class_name,window_size):
    if class_name=="still":
        ax=np.random.normal(0.0,0.03,window_size)
        ay=np.random.normal(0.0,0.03,window_size)
        az=np.random.normal(1.0,0.03,window_size)
    elif class_name=="left_tilt":
        ax=np.random.normal(-0.7,0.05,window_size)
        ay=np.random.normal(0.0,0.03,window_size)
        az=np.random.normal(0.7,0.05,window_size)
    elif class_name=="right_tilt":
        ax=np.random.normal(0.7,0.05,window_size)
        ay=np.random.normal(0.0,0.03,window_size)
        az=np.random.normal(0.7,0.05,window_size)
    elif class_name=="shake":
        ax=np.random.normal(0,0.9,window_size)
        ay=np.random.normal(0,0.9,window_size)
        az=np.random.normal(0.2,0.9,window_size)
    else:
        raise ValueError("Unknown class name")
    
    window=np.column_stack((ax,ay,az))
    return window

In [10]:
def extract_features(window):
    window=np.array(window,dtype=np.float32)

    ax=window[:,0]
    ay=window[:,1]
    az=window[:,2]

    features=np.array([
        np.mean(ax), np.std(ax), np.min(ax), np.max(ax),
        np.mean(ay), np.std(ay), np.min(ay), np.max(ay),
        np.mean(az), np.std(az), np.min(az), np.max(az),
    ], dtype=np.float32)
    
    return features

In [11]:
data = []
labels = []
raw_windows = []    

In [12]:
for label_idx,class_name in enumerate(class_names):
    for _ in range(samples_per_class):
        window=generate_accel_window(class_name,window_size)
        features_vec=extract_features(window)
        data.append(features_vec)
        labels.append(label_idx)
        raw_windows.append(window.tolist())

In [13]:
X=np.array(data,dtype=np.float32)
y=np.array(labels,dtype=np.int32)

In [14]:
print("Data shape:", X.shape)
print("Labels shape:", y.shape)
print("Expected Shape:", (len(class_names)*samples_per_class, len(features)))

Data shape: (4000, 12)
Labels shape: (4000,)
Expected Shape: (4000, 12)


In [15]:
df_features=pd.DataFrame(X,columns=features)
df_features['label']=y

In [16]:
df_features.head()

,ax_mean,ax_std,ax_min,axmax,ay_mean,ay_std,ay_min,aymax,az_mean,az_std,az_min,azmax,label
0,-0.005139,0.028072,-0.057398,0.047376,-0.007979,0.028306,-0.058790,0.055568,0.999199,0.024002,0.947109,1.031714,0
1,-0.000942,0.032518,-0.078592,0.046939,-0.000716,0.020200,-0.043905,0.044337,1.001326,0.029903,0.942437,1.073897,0
2,-0.003396,0.031279,-0.046520,0.065714,0.001422,0.026408,-0.048224,0.055973,1.010246,0.029375,0.970760,1.081605,0
3,-0.006253,0.021488,-0.045445,0.025692,0.012718,0.031513,-0.041330,0.115582,0.998648,0.034682,0.939246,1.069440,0
4,0.003835,0.030189,-0.045581,0.063665,-0.010375,0.032719,-0.097238,0.048972,1.004908,0.027727,0.941437,1.063991,0


In [17]:
X_train,X_test,y_train,y_test,raw_train,raw_test=train_test_split(X,y,raw_windows,test_size=0.2,random_state=42,stratify=y)

In [18]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [19]:
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(features),)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

In [20]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            36 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 380 (1.48 KB)

 Trainable params: 380 (1.48 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
history=model.fit(X_train_scaled,y_train,validation_data=(X_test_scaled,y_test),epochs=30,batch_size=32)    

Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.5181 - loss: 1.2736 - val_accuracy: 0.7500 - val_loss: 0.9801
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7500 - loss: 0.7932 - val_accuracy: 0.7500 - val_loss: 0.6430
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7500 - loss: 0.5485 - val_accuracy: 0.7500 - val_loss: 0.4670
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7500 - loss: 0.4070 - val_accuracy: 0.7500 - val_loss: 0.3513
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7891 - loss: 0.3048 - val_accuracy: 0.9912 - val_loss: 0.2597
Epoch 6/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9991 - loss: 0.2190 - val_accuracy: 1.0000 - val_loss: 0.1769
Epoch 7/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.1335 - val_accuracy: 1.0000 - val_loss: 0.0897
Epoch 8/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0584 - val_accuracy: 1

In [23]:
y_pred_prob=model.predict(X_test_scaled)
y_pred=np.argmax(y_pred_prob, axis=1)

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  


In [24]:
float_model_acc=accuracy_score(y_test, y_pred)
print(f"Float model accuracy: {float_model_acc:.4f}")

Float model accuracy: 1.0000


In [25]:
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Confusion matrix:
[[200   0   0   0]
 [  0 200   0   0]
 [  0   0 200   0]
 [  0   0   0 200]]


In [26]:
scaler_params={
    'mean': scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist()
}

preprocess_params={
    'window_size': window_size,
    'features': features,
    'axes': ['ax', 'ay', 'az']
}

In [27]:
with open("scaler_params.json", "w") as f:
    json.dump(scaler_params, f, indent=4)

with open("preprocess_params.json", "w") as f:
    json.dump(preprocess_params, f, indent=4)

with open("label_map.json", "w") as f:
    json.dump(label_map, f, indent=4)

In [28]:
sample_test_window=raw_test[:20]
with open("sample_test_window.json", "w") as f:
    json.dump(sample_test_window, f, indent=4)

In [29]:
sample_test_window[0]

[[0.0068118915943244885, 0.018683073756826502, 0.988497461740932],
 [-0.002316651347952771, 0.024700704884315177, 0.9901709968807012],
 [0.038854456325860286, -0.059387828275575844, 0.9781501219473477],
 [0.01214824061446029, -0.012403164071295308, 1.0230458238431754],
 [-0.025159729125617543, 0.02849101152833609, 0.9723853363678127],
 [-0.02042726487407084, 0.008234050069096947, 1.025501091855687],
 [0.020459798977624365, -0.035895451444527986, 1.0062355971441295],
 [0.03125198557963791, -0.07290367127182068, 0.9986486656463454],
 [0.016550406967679086, -0.010973422359060191, 0.980298688006706],
 [0.056906662436162436, -0.0021475707868163713, 0.9593337441650762],
 [0.04905021497194256, -0.02651768849485036, 0.9805704173086587],
 [0.013231075045522886, 0.004292548110839832, 0.9900997939412534],
 [-0.025157419784445068, 0.01519160241698684, 0.9289292706525625],
 [-0.03929962535195303, 0.020120615121786985, 0.9940793453399601],
 [-0.011621452350441444, -0.07067756774796967, 1.02773072104

In [50]:
def representative_dataset():
    for i in range(min(500, len(X_train_scaled))):
        sample = X_train_scaled[i].reshape(1, 12).astype(np.float32)
        yield [sample]

converter=tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations=[tf.lite.Optimize.DEFAULT]
converter.representative_dataset=representative_dataset
converter.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type=tf.uint8
converter.inference_output_type=tf.uint8

In [51]:
tflite_model = converter.convert()

Saved artifact at '/tmp/tmpgqfb4sl2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 12), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  134562394867856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394869008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394867664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394866320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394869584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394866512: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [52]:
with open("model_int8.tflite", "wb") as f:
    f.write(tflite_model)

In [54]:
# save the TFLite model without quantization
converter=tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model_float = converter.convert()

with open("model_float.tflite", "wb") as f:
    f.write(tflite_model_float)

Saved artifact at '/tmp/tmpvior1dho'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 12), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  134562394867856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394869008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394867664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394866320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394869584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134562394866512: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [55]:
# get the size of the TFLite model without quantization
model_size_float = os.path.getsize("model_float.tflite")
print(f"TFLite model size (float): {model_size_float/1024} KB")

TFLite model size (float): 3.5546875 KB


In [56]:
# get the size of the TFLite model
model_size = os.path.getsize("model_int8.tflite")
print(f"TFLite model size: {model_size/1024} KB")

TFLite model size: 3.875 KB


In [57]:
interpreter = tf.lite.Interpreter(model_path="model_int8.tflite")
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
print("Input dtype:", input_details['dtype'])
print("Output dtype:", output_details['dtype'])
print("Input scale/zero_point:", input_details['quantization'])
print("Output scale/zero_point:", output_details['quantization'])

Input dtype: <class 'numpy.uint8'>
Output dtype: <class 'numpy.uint8'>
Input scale/zero_point: (0.036785513162612915, 128)
Output scale/zero_point: (0.00390625, 0)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [40]:
def run_tflite_inference(interpreter, X_scaled):
    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    in_scale, in_zero  = input_details['quantization']
    out_scale, out_zero = output_details['quantization']

    preds = []
    for sample in X_scaled:

        x = (sample / in_scale + in_zero).astype(np.uint8).reshape(1, 12)
        interpreter.set_tensor(input_details['index'], x)
        interpreter.invoke()


        out = interpreter.get_tensor(output_details['index'])[0]
        out_float = (out.astype(np.float32) - out_zero) * out_scale
        preds.append(np.argmax(out_float))

    return np.array(preds)


y_pred_tflite = run_tflite_inference(interpreter, X_test_scaled)

In [41]:
from sklearn.metrics import accuracy_score

tflite_acc = accuracy_score(y_test, y_pred_tflite)
print(f"Float model accuracy:     {float_model_acc:.4f}")
print(f"INT8 TFLite accuracy:     {tflite_acc:.4f}")
print(f"Accuracy drop:            {float_model_acc - tflite_acc:.4f}")

Float model accuracy:     1.0000
INT8 TFLite accuracy:     1.0000
Accuracy drop:            0.0000
